# 04주차 실습: 분포·가설검정·신뢰구간

강의: [PDF](../pdf/week04_statistics_fundamentals.pdf) · **PDF 실제 페이지 4–13** (인쇄 번호와 다를 수 있음)

## 학습 목표
PDF/CDF를 구분하고 효과 크기·불확실성·상관관계를 함께 해석한다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## 기술 통계와 확률 분포

In [ ]:
from scipy import stats
rng = np.random.default_rng(42)
samples = {'Normal': rng.normal(0, 1, 2400), 'Exponential': rng.exponential(1, 2400),
           'Pareto': rng.pareto(2, 2400) + 1, 'Poisson': rng.poisson(3, 2400)}
display(pd.DataFrame(samples).describe())
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for (name, values), ax in zip(samples.items(), axes.flat):
    x = np.sort(values)
    ax.step(x, np.arange(1, len(x)+1)/len(x), where='post'); ax.set(title=name, ylabel='ECDF')
plt.tight_layout(); plt.show()
print('Normal P(-1 <= X <= 1):', stats.norm.cdf(1)-stats.norm.cdf(-1))

## Welch 검정과 평균 차이 구간

In [ ]:
a, b = rng.normal(10, 2, 1000), rng.normal(10.3, 3, 1000)
test = stats.ttest_ind(b, a, equal_var=False)
v1, v2 = b.var(ddof=1)/len(b), a.var(ddof=1)/len(a)
se = np.sqrt(v1+v2)
dof = (v1+v2)**2 / (v1**2/(len(b)-1) + v2**2/(len(a)-1))
delta = b.mean()-a.mean()
ci = delta + np.array([-1,1])*stats.t.ppf(.975,dof)*se
out = output_dir('week04')
save_json(out/'inference.json', {'difference': delta, 'pvalue': test.pvalue, 'ci95': ci.tolist()})
print('차이, p, 95% CI:', delta, test.pvalue, ci)
z = rng.normal(size=2400); x = z+rng.normal(size=2400); y=z+rng.normal(size=2400)
print('공통 원인 z로 만든 x/y 상관:', stats.pearsonr(x,y).statistic)
assert ci[0] < ci[1]

## 확장 과제 및 제출
기본 실행은 합성 데이터입니다. PDF 과제 제출 시 출처가 있는 실제 관측치 1,000개 이상으로 교체하여 histogram/PDF·ECDF·QQ plot과 통계 리포트를 작성하세요. p-value를 귀무가설이 참일 확률로 해석하지 마세요.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.